In [1]:
# ═══════════════════════════════════════
# TRY 2 — STANDARD SESSION OPENER
# ═══════════════════════════════════════

import json, os, pickle, random, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from sklearn.metrics import precision_score, recall_score, f1_score
from xgboost import XGBClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = "/kaggle/input/datasets/aadityaupadhyay1353"

CFG_PATH = f"{BASE}/cps-config/config.json"

with open(CFG_PATH) as f:
    CFG = json.load(f)

TRAIN_IDS = CFG["engine_partition"]["train"]
VAL_IDS = CFG["engine_partition"]["validation"]

OUT = "/kaggle/working/"
os.makedirs(OUT, exist_ok=True)

print("TRY 2 session ready")

TRY 2 session ready


In [2]:
df = pd.read_csv(
    f"{BASE}/v2-phase3-features/v2_phase3_feature_dataset.csv"
)

with open(f"{BASE}/v2-phase3-features/v2_feature_cols.json") as f:
    feature_cols = json.load(f)

scaler = pickle.load(
    open(f"{BASE}/v2-phase3-features/v2_scaler.pkl","rb")
)

assert "max_cycle" not in feature_cols

print("Dataset shape:", df.shape)
print("Features:", len(feature_cols))

Dataset shape: (20631, 65)
Features: 56


In [3]:
top_models = pd.read_csv(
    f"{BASE}/v2-baseline-results/v2_top_models.csv"
)

best = top_models.iloc[0]

BEST_W = int(best["window"])

LABEL_COL = f"label_w{BEST_W}"

print("Best window:", BEST_W)
print("Label column:", LABEL_COL)

Best window: 20
Label column: label_w20


In [4]:
train_df = df[df["engine_id"].isin(TRAIN_IDS)]
val_df = df[df["engine_id"].isin(VAL_IDS)]

X_val = scaler.transform(val_df[feature_cols])
y_val = val_df[LABEL_COL].values

print("Train rows:", train_df.shape)
print("Val rows:", val_df.shape)

Train rows: (14130, 65)
Val rows: (3210, 65)


In [5]:
NODE_PARTITION = CFG["node_partition"]

print("Node structure:")
print(NODE_PARTITION)

Node structure:
{'node_A': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], 'node_B': [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28], 'node_C': [29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42], 'node_D': [43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56], 'node_E': [57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70]}


In [6]:
node_models = {}
node_f1_scores = {}
all_predictions = []

for node_id, engines in NODE_PARTITION.items():

    print("\nTraining", node_id)

    node_train = train_df[train_df["engine_id"].isin(engines)]

    X_node = scaler.transform(node_train[feature_cols])
    y_node = node_train[LABEL_COL].values

    neg, pos = np.bincount(y_node)

    scale_pos_weight = neg / pos

    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=SEED,
        verbosity=0
    )

    model.fit(X_node, y_node)

    node_models[node_id] = model

    preds_val = model.predict(X_val)
    probs_val = model.predict_proba(X_val)[:,1]

    f1 = f1_score(y_val, preds_val)

    node_f1_scores[node_id] = f1

    print(node_id, "F1:", round(f1,4))

    chunk = val_df[["engine_id","cycle",LABEL_COL]].copy()

    chunk["node_id"] = node_id
    chunk["prediction"] = preds_val
    chunk["prob"] = probs_val

    chunk.rename(columns={LABEL_COL:"true_label"}, inplace=True)

    all_predictions.append(chunk)


Training node_A
node_A F1: 0.8576

Training node_B
node_B F1: 0.8418

Training node_C
node_C F1: 0.7867

Training node_D
node_D F1: 0.8732

Training node_E
node_E F1: 0.7437


In [7]:
pred_df = pd.concat(all_predictions)

pred_df.to_csv(
    f"{OUT}v2_node_predictions_val.csv",
    index=False
)

print("Saved node predictions")

Saved node predictions


In [8]:
total_f1 = sum(node_f1_scores.values())

weights = []

for node_id in node_f1_scores:

    w = node_f1_scores[node_id] / total_f1

    weights.append({
        "node_id": node_id,
        "f1": node_f1_scores[node_id],
        "weight": w
    })

weights_df = pd.DataFrame(weights)

weights_df.to_csv(
    f"{OUT}v2_node_f1_weights.csv",
    index=False
)

print(weights_df)

  node_id        f1    weight
0  node_A  0.857562  0.209011
1  node_B  0.841791  0.205167
2  node_C  0.786704  0.191741
3  node_D  0.873156  0.212812
4  node_E  0.743733  0.181268


In [9]:
for node_id, model in node_models.items():

    pickle.dump(
        model,
        open(f"{OUT}v2_{node_id}_xgb.pkl","wb")
    )

print("Saved node models")

Saved node models


In [10]:
!ls -lh /kaggle/working/

total 1.8M
---------- 1 root root  16K Apr 12 13:07 __notebook__.ipynb
-rw-r--r-- 1 root root 243K Apr 12 13:07 v2_node_A_xgb.pkl
-rw-r--r-- 1 root root 261K Apr 12 13:07 v2_node_B_xgb.pkl
-rw-r--r-- 1 root root 246K Apr 12 13:07 v2_node_C_xgb.pkl
-rw-r--r-- 1 root root 252K Apr 12 13:07 v2_node_D_xgb.pkl
-rw-r--r-- 1 root root 263K Apr 12 13:07 v2_node_E_xgb.pkl
-rw-r--r-- 1 root root  247 Apr 12 13:07 v2_node_f1_weights.csv
-rw-r--r-- 1 root root 481K Apr 12 13:07 v2_node_predictions_val.csv
